# 宮崎空港 Traffic Pattern and Independent Turn Paths

RJFM の RWY09 / RWY27、LEFT / RIGHTごとに、Circle／270を含まない通常場周と、Before Downwind、Middle Downwind、Before BaseのCircle、Before Downwind/Baseの270を独立Pathとして生成します。task-provided Short Downwindと110 kt・22° BankのCircleも同時に表示・出力します。各pathは風に依存しない幾何学的な `ReferencePath` です。

> ARP は参照・検算専用です。すべての geometry は両 threshold の中点である **RWY Center Point** と True Bearing を基準に計算します。

## 1. AirportSpec と出力先

In [ ]:
from pathlib import Path
import math
import os

from IPython.display import display
from matplotlib import pyplot as plt

from sr22_course_simulator.data.airports import RJFM
from sr22_course_simulator.examples.miyazaki_traffic_patterns import (
    RJFM_COMBINED_PATTERN_FILENAME,
    RJFM_PATTERN_FILENAMES,
    RJFM_SHORT_DOWNWIND_CIRCLE_FILENAME,
    RJFM_SHORT_DOWNWIND_COMBINED_FILENAME,
    RJFM_SHORT_DOWNWIND_ENTRY_RWY27_FILENAME,
    RJFM_SHORT_DOWNWIND_FILENAME,
    build_rjfm_short_downwind_paths,
    build_rjfm_traffic_pattern_components,
    rjfm_short_downwind_circle_spec,
    rjfm_traffic_pattern_specs,
    write_rjfm_short_downwind_kmls,
    write_rjfm_traffic_pattern_kmls,
)
from sr22_course_simulator.units import metres_to_nautical_miles

default_artifact_root = (
    Path.cwd().parent / 'artifacts'
    if Path.cwd().name == 'notebooks'
    else Path.cwd() / 'artifacts'
)
artifact_root = Path(os.environ.get('SR22_ARTIFACT_DIR', default_artifact_root)).resolve()
output_dir = artifact_root / 'traffic-patterns'
output_dir.mkdir(parents=True, exist_ok=True)
display(RJFM)
print(f'KML output: {output_dir}')

## 2. RWY True Bearing と threshold

`threshold_a` は着陸 threshold、`threshold_b` は reciprocal / departure-end threshold です。KML の方位計算には True Bearing だけを使います。

In [ ]:
runway_rows = [
    {
        'designation': runway.designation,
        'true_bearing_deg': runway.true_bearing_deg,
        'threshold_a': runway.threshold_a,
        'threshold_b': runway.threshold_b,
        'threshold_elevation_a_ft': runway.threshold_elevation_a_ft,
        'threshold_elevation_b_ft': runway.threshold_elevation_b_ft,
        'measured_length_m': runway.measured_length_m,
    }
    for runway in RJFM.runways
]
display(runway_rows)
print(f'MAG VAR at 2026.0 (reference only): {RJFM.variation_at(2026.0):.2f} deg')

## 3. RWY Center Point

`center_point = (threshold_a + threshold_b) / 2` を deterministic に計算します。ARP をこの計算へ渡しません。

In [ ]:
for runway in RJFM.runways:
    print(f'RWY{runway.designation} center: {runway.center_point}')
assert RJFM.runway('09').center_point == RJFM.runway('27').center_point

## 4. Pattern parameters

Downwind axisは中心線から1.5 NM、Crosswind/Base axisは各THRから1.2 NMです。純粋な通常場周は110 kt・22° Bankの90° Downwind/Base Turnで構成し、Base Turn開始と同時に降下します。3つの `MAKE_CIRCLE_*` とBefore Downwind/Baseの `MAKE_270_*` は互いに独立した追加Pathとして選択でき、Circle ON / 270 OFFも有効です。Before Downwind/BaseのCircleは270のON/OFFにかかわらず潜在270の開始座標・進入Headingから始まり、270 Pathだけが通常場周のbranch→接続線→270° arc→接続線→mergeを含みます。Before Base 270は通常22° Base Turnと同じ水平沿程長だけmerge手前で降下します。Short Downwind本体とRWY27 Entryはtask-provided KML座標を変更せず、Circle半径だけ110 kt・22° Bankで再計算します。

In [ ]:
PATTERN_ALTITUDE_FT = 1000.0
DOWNWIND_OFFSET_NM = 1.5
CROSSWIND_BASE_EXTENSION_NM = 1.2
MAKE_CIRCLE_BEFORE_DOWNWIND = True
# Circleと270は独立Pathです。任意のON/OFF組み合わせが有効です。
MAKE_270_BEFORE_DOWNWIND = True
MAKE_CIRCLE_MIDDLE_DOWNWIND = True
MAKE_CIRCLE_BEFORE_BASE = True
MAKE_270_BEFORE_BASE = True
TRUE_AIRSPEED_KT = 110.0
NORMAL_BANK_DEG = 22.0
DOWNWIND_TURN_BANK_DEG = 22.0
BASE_TURN_BANK_DEG = 22.0
FINAL_BANK_DEG = 25.0
ROLL_RATE_DEG_S = 10.0
GLIDE_PATH_DEG = 3.0
SAMPLE_INTERVAL_S = 0.25
SHORT_DOWNWIND_CIRCLE_TRUE_AIRSPEED_KT = 110.0
SHORT_DOWNWIND_CIRCLE_BANK_DEG = 22.0
# KML color order: AABBGGRR (line alpha=ff, fill alpha=1a ≈ 10%)
RWY09_LINE_COLOR = 'ffffff00'  # cyan
RWY09_FILL_COLOR = '1affffcc'  # pale cyan
RWY27_LINE_COLOR = 'ff00aaff'  # orange
RWY27_FILL_COLOR = '1a80d4ff'  # pale orange
MAGNETIC_REFERENCE_YEAR = 2026.0

In [ ]:
pattern_specs = rjfm_traffic_pattern_specs(
    altitude_ft=PATTERN_ALTITUDE_FT,
    downwind_offset_nm=DOWNWIND_OFFSET_NM,
    crosswind_base_extension_nm=CROSSWIND_BASE_EXTENSION_NM,
    true_airspeed_kt=TRUE_AIRSPEED_KT,
    normal_bank_deg=NORMAL_BANK_DEG,
    downwind_turn_bank_deg=DOWNWIND_TURN_BANK_DEG,
    final_bank_deg=FINAL_BANK_DEG,
    base_turn_bank_deg=BASE_TURN_BANK_DEG,
    roll_rate_deg_s=ROLL_RATE_DEG_S,
    glide_path_deg=GLIDE_PATH_DEG,
    sample_interval_s=SAMPLE_INTERVAL_S,
    make_circle_before_downwind=MAKE_CIRCLE_BEFORE_DOWNWIND,
    make_circle_middle_downwind=MAKE_CIRCLE_MIDDLE_DOWNWIND,
    make_circle_before_base=MAKE_CIRCLE_BEFORE_BASE,
    make_270_before_downwind=MAKE_270_BEFORE_DOWNWIND,
    make_270_before_base=MAKE_270_BEFORE_BASE,
)
display(pattern_specs)
sample_spec = pattern_specs[0]
print(f'Normal turn radius: {metres_to_nautical_miles(sample_spec.normal_turn_radius_m):.3f} NM')
print(f'Final turn radius: {metres_to_nautical_miles(sample_spec.final_turn_radius_m):.3f} NM')
print(f'Ordinary Downwind turn radius: {metres_to_nautical_miles(sample_spec.downwind_turn_radius_m):.3f} NM')
print(f'Ordinary Base turn radius: {metres_to_nautical_miles(sample_spec.base_turn_radius_m):.3f} NM')
short_circle_spec = rjfm_short_downwind_circle_spec(
    true_airspeed_kt=SHORT_DOWNWIND_CIRCLE_TRUE_AIRSPEED_KT,
    bank_deg=SHORT_DOWNWIND_CIRCLE_BANK_DEG,
)
print(f'Short Downwind circle radius: {metres_to_nautical_miles(short_circle_spec.radius_m):.3f} NM')
print(f'Magnetic reference at {MAGNETIC_REFERENCE_YEAR}: {RJFM.variation_at(MAGNETIC_REFERENCE_YEAR):.2f} deg (not used in geometry)')

## 5. 通常場周と独立Circle／270 Pathを生成

RWY／LEFT・RIGHTごとに、Circleも270も含まない通常場周を必ず1本生成します。ONにした各Circleと270は通常場周へ連結せず、独立ReferencePathとして追加します。Short Downwind本体とCircleも別ReferencePathです。

In [ ]:
pattern_components = build_rjfm_traffic_pattern_components(
    altitude_ft=PATTERN_ALTITUDE_FT,
    downwind_offset_nm=DOWNWIND_OFFSET_NM,
    crosswind_base_extension_nm=CROSSWIND_BASE_EXTENSION_NM,
    true_airspeed_kt=TRUE_AIRSPEED_KT,
    normal_bank_deg=NORMAL_BANK_DEG,
    downwind_turn_bank_deg=DOWNWIND_TURN_BANK_DEG,
    final_bank_deg=FINAL_BANK_DEG,
    base_turn_bank_deg=BASE_TURN_BANK_DEG,
    roll_rate_deg_s=ROLL_RATE_DEG_S,
    glide_path_deg=GLIDE_PATH_DEG,
    sample_interval_s=SAMPLE_INTERVAL_S,
    make_circle_before_downwind=MAKE_CIRCLE_BEFORE_DOWNWIND,
    make_circle_middle_downwind=MAKE_CIRCLE_MIDDLE_DOWNWIND,
    make_circle_before_base=MAKE_CIRCLE_BEFORE_BASE,
    make_270_before_downwind=MAKE_270_BEFORE_DOWNWIND,
    make_270_before_base=MAKE_270_BEFORE_BASE,
)
short_downwind_paths = build_rjfm_short_downwind_paths(
    circle_true_airspeed_kt=SHORT_DOWNWIND_CIRCLE_TRUE_AIRSPEED_KT,
    circle_bank_deg=SHORT_DOWNWIND_CIRCLE_BANK_DEG,
)
for path in pattern_components:
    labels = [point.label for point in path.points() if point.label]
    print(path.name, len(path.points()), labels)
for path in short_downwind_paths:
    print(path.name, len(path.points()))
expected_component_count = 4 * (1 + sum((
    MAKE_CIRCLE_BEFORE_DOWNWIND,
    MAKE_270_BEFORE_DOWNWIND,
    MAKE_CIRCLE_MIDDLE_DOWNWIND,
    MAKE_CIRCLE_BEFORE_BASE,
    MAKE_270_BEFORE_BASE,
)))
assert len(pattern_components) == expected_component_count
assert len(short_downwind_paths) == 3

## 6. 簡易可視化

In [ ]:
figure, axes = plt.subplots(figsize=(10, 8))
for path in pattern_components:
    longitudes = [point.position.longitude_deg for point in path.points()]
    latitudes = [point.position.latitude_deg for point in path.points()]
    axes.plot(longitudes, latitudes, label=path.name)
for path in short_downwind_paths:
    longitudes = [point.position.longitude_deg for point in path.points()]
    latitudes = [point.position.latitude_deg for point in path.points()]
    axes.plot(longitudes, latitudes, linestyle='--', label=path.name)
center = RJFM.runway('09').center_point
axes.scatter([center.longitude_deg], [center.latitude_deg], marker='x', s=100, color='black', label='RWY Center Point')
axes.scatter([RJFM.reference_point.longitude_deg], [RJFM.reference_point.latitude_deg], marker='+', s=100, color='gray', label='ARP (reference only)')
axes.set_xlabel('Longitude [deg]')
axes.set_ylabel('Latitude [deg]')
axes.set_title('RJFM Traffic Pattern and Independent Circle / 270 Paths')
axes.set_aspect(1.0 / math.cos(math.radians(center.latitude_deg)), adjustable='datalim')
axes.grid(True)
axes.legend()
plt.show()

## 7. KML 出力

通常場周4個の各KMLには、Circle／270なしの場周本体と、ONにした各Circle／270を別Placemarkとして収録します。全場周の結合KMLでも分離を維持します。Google Earth向けにabsolute altitude、線幅1.0・不透明度100%、地面までのfill 10%、枠線OFFを使用します。RWY 09はシアン、RWY 27はオレンジを初期値とし、parameter cellのKML AABBGGRR色で変更できます。Short Downwind単独KMLは共通色、RWY27 Entry／CircleとEntry・Path・Circle結合KMLはRWY 27色です。

In [ ]:
written = write_rjfm_traffic_pattern_kmls(
    output_dir,
    altitude_ft=PATTERN_ALTITUDE_FT,
    downwind_offset_nm=DOWNWIND_OFFSET_NM,
    crosswind_base_extension_nm=CROSSWIND_BASE_EXTENSION_NM,
    true_airspeed_kt=TRUE_AIRSPEED_KT,
    normal_bank_deg=NORMAL_BANK_DEG,
    downwind_turn_bank_deg=DOWNWIND_TURN_BANK_DEG,
    final_bank_deg=FINAL_BANK_DEG,
    base_turn_bank_deg=BASE_TURN_BANK_DEG,
    roll_rate_deg_s=ROLL_RATE_DEG_S,
    glide_path_deg=GLIDE_PATH_DEG,
    sample_interval_s=SAMPLE_INTERVAL_S,
    make_circle_before_downwind=MAKE_CIRCLE_BEFORE_DOWNWIND,
    make_circle_middle_downwind=MAKE_CIRCLE_MIDDLE_DOWNWIND,
    make_circle_before_base=MAKE_CIRCLE_BEFORE_BASE,
    make_270_before_downwind=MAKE_270_BEFORE_DOWNWIND,
    make_270_before_base=MAKE_270_BEFORE_BASE,
    rwy09_line_color=RWY09_LINE_COLOR,
    rwy09_fill_color=RWY09_FILL_COLOR,
    rwy27_line_color=RWY27_LINE_COLOR,
    rwy27_fill_color=RWY27_FILL_COLOR,
)
written += write_rjfm_short_downwind_kmls(
    output_dir,
    circle_true_airspeed_kt=SHORT_DOWNWIND_CIRCLE_TRUE_AIRSPEED_KT,
    circle_bank_deg=SHORT_DOWNWIND_CIRCLE_BANK_DEG,
    rwy27_line_color=RWY27_LINE_COLOR,
    rwy27_fill_color=RWY27_FILL_COLOR,
)
expected_names = {
    *RJFM_PATTERN_FILENAMES,
    RJFM_COMBINED_PATTERN_FILENAME,
    RJFM_SHORT_DOWNWIND_FILENAME,
    RJFM_SHORT_DOWNWIND_CIRCLE_FILENAME,
    RJFM_SHORT_DOWNWIND_ENTRY_RWY27_FILENAME,
    RJFM_SHORT_DOWNWIND_COMBINED_FILENAME,
}
assert {path.name for path in written} == expected_names
assert all(path.is_file() for path in written)
for path in written:
    print(path)
